In [1]:
test_cases = [
    {
        "agent": "Document Agent",
        "question": "What is the required inspection interval?",
        "expected_answer": "Every 30 days",
        "actual_answer": "Every 30 days",
        "citation": "Section 4, Page 12",
        "expected_citation": "Section 4, Page 12"
    },
    {
        "agent": "Document Agent",
        "question": "What is the required inspection interval?",
        "expected_answer": "Every 30 days",
        "actual_answer": "Every 60 days",
        "citation": "Section 4, Page 12",
        "expected_citation": "Section 4, Page 12"
    },
    {
        "agent": "Policy Agent",
        "question": "Does this action need approval?",
        "expected_answer": "Approval required",
        "actual_answer": "Approval required",
        "citation": "",
        "expected_citation": ""
    },
    {
        "agent": "Reporting Agent",
        "question": "Summarize the project status.",
        "expected_answer": "Project is delayed",
        "actual_answer": "Project is delayed",
        "citation": "",
        "expected_citation": ""
    }
]

print("Test cases loaded:", len(test_cases))

Test cases loaded: 4


In [2]:
def check_agent(test):
    problems = []

    # Check whether the answer matches the expected answer
    if test["actual_answer"].strip().lower() != \
       test["expected_answer"].strip().lower():
        problems.append("Answer mismatch")

    # Check whether a required citation is present
    expected_citation = test["expected_citation"]
    actual_citation = test["citation"]

    if expected_citation and not actual_citation:
        problems.append("Missing citation")
    elif expected_citation and actual_citation != expected_citation:
        problems.append("Citation mismatch")

    return {
        "agent": test["agent"],
        "status": "FLAGGED" if problems else "PASS",
        "problems": problems
    }


results = [check_agent(test) for test in test_cases]

for result in results:
    print(result)

{'agent': 'Document Agent', 'status': 'PASS', 'problems': []}
{'agent': 'Document Agent', 'status': 'FLAGGED', 'problems': ['Answer mismatch']}
{'agent': 'Policy Agent', 'status': 'PASS', 'problems': []}
{'agent': 'Reporting Agent', 'status': 'PASS', 'problems': []}


In [3]:

from datetime import datetime
import pandas as pd

def evaluate_agent(test):
    issues = []

    # Check answer accuracy against the expected answer
    if test["actual_answer"].strip().lower() != \
       test["expected_answer"].strip().lower():
        issues.append("Answer mismatch")

    # Check citations
    expected = test["expected_citation"]
    actual = test["citation"]

    if expected and not actual:
        issues.append("Missing citation")
    elif expected and actual != expected:
        issues.append("Citation mismatch")

    # Assign severity
    if "Answer mismatch" in issues:
        severity = "HIGH"
    elif issues:
        severity = "MEDIUM"
    else:
        severity = "NONE"

    return {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "agent": test["agent"],
        "question": test["question"],
        "status": "FLAGGED" if issues else "PASS",
        "severity": severity,
        "issues": ", ".join(issues) if issues else "None"
    }


# Evaluate every test case from Step 1
audit_log = pd.DataFrame(
    [evaluate_agent(test) for test in test_cases]
)

display(audit_log)

,timestamp,agent,question,status,severity,issues
0,2026-10-10T08:00:05,Document Agent,What is the required inspection interval?,PASS,NONE,None
1,2026-10-10T08:00:05,Document Agent,What is the required inspection interval?,FLAGGED,HIGH,Answer mismatch
2,2026-10-10T08:00:05,Policy Agent,Does this action need approval?,PASS,NONE,None
3,2026-10-10T08:00:05,Reporting Agent,Summarize the project status.,PASS,NONE,None


In [4]:
total = len(audit_log)
flagged = (audit_log["status"] == "FLAGGED").sum()
passed = (audit_log["status"] == "PASS").sum()

print("Total tests:", total)
print("Passed:", passed)
print("Flagged:", flagged)
print("Flag rate:", round(flagged / total * 100, 1), "%")

print("\nIssues by severity:")
print(audit_log["severity"].value_counts())

Total tests: 4
Passed: 3
Flagged: 1
Flag rate: 25.0 %

Issues by severity:
severity
NONE    3
HIGH    1
Name: count, dtype: int64


In [5]:
evaluation_cases = [
    {
        "name": "Correct answer",
        "expected_answer": "Every 30 days",
        "actual_answer": "Every 30 days",
        "expected_citation": "Page 12",
        "citation": "Page 12",
        "known_problem": False
    },
    {
        "name": "Incorrect answer",
        "expected_answer": "Every 30 days",
        "actual_answer": "Every 60 days",
        "expected_citation": "Page 12",
        "citation": "Page 12",
        "known_problem": True
    },
    {
        "name": "Missing citation",
        "expected_answer": "Approval required",
        "actual_answer": "Approval required",
        "expected_citation": "Policy 4",
        "citation": "",
        "known_problem": True
    },
    {
        "name": "Wrong citation",
        "expected_answer": "Every 30 days",
        "actual_answer": "Every 30 days",
        "expected_citation": "Page 12",
        "citation": "Page 20",
        "known_problem": True
    }
]

def watchdog_flag(case):
    answer_wrong = (
        case["actual_answer"].strip().lower()
        != case["expected_answer"].strip().lower()
    )

    citation_wrong = (
        bool(case["expected_citation"])
        and case["citation"] != case["expected_citation"]
    )

    return answer_wrong or citation_wrong


for case in evaluation_cases:
    predicted_problem = watchdog_flag(case)
    actual_problem = case["known_problem"]

    if predicted_problem and actual_problem:
        result = "TRUE POSITIVE"
    elif not predicted_problem and not actual_problem:
        result = "TRUE NEGATIVE"
    elif predicted_problem and not actual_problem:
        result = "FALSE POSITIVE"
    else:
        result = "FALSE NEGATIVE"

    print(case["name"], "->", result)

Correct answer -> TRUE NEGATIVE
Incorrect answer -> TRUE POSITIVE
Missing citation -> TRUE POSITIVE
Wrong citation -> TRUE POSITIVE


In [6]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

actual = [case["known_problem"] for case in evaluation_cases]
predicted = [watchdog_flag(case) for case in evaluation_cases]

print("Confusion matrix:")
print(confusion_matrix(actual, predicted, labels=[False, True]))

print("\nClassification report:")
print(classification_report(
    actual,
    predicted,
    labels=[False, True],
    target_names=["No problem", "Problem"],
    zero_division=0
))

Confusion matrix:
[[1 0]
 [0 3]]

Classification report:
              precision    recall  f1-score   support

  No problem       1.00      1.00      1.00         1
     Problem       1.00      1.00      1.00         3

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



In [7]:
import pandas as pd
from datetime import datetime

incidents = []

for case in evaluation_cases:
    answer_wrong = (
        case["actual_answer"].strip().lower()
        != case["expected_answer"].strip().lower()
    )

    citation_wrong = (
        bool(case["expected_citation"])
        and case["citation"] != case["expected_citation"]
    )

    issues = []

    if answer_wrong:
        issues.append("Answer mismatch")

    if citation_wrong:
        if not case["citation"]:
            issues.append("Missing citation")
        else:
            issues.append("Incorrect citation")

    if issues:
        severity = "HIGH" if answer_wrong else "MEDIUM"

        incidents.append({
            "Incident_ID": f"AI-{len(incidents) + 1:03d}",
            "Timestamp": datetime.now().isoformat(timespec="seconds"),
            "Agent_Test": case["name"],
            "Severity": severity,
            "Issue": ", ".join(issues),
            "Status": "Open",
            "Recommended_Action": (
                "Human review required; verify answer and source."
            )
        })

incident_log = pd.DataFrame(incidents)

display(incident_log)

,Incident_ID,Timestamp,Agent_Test,Severity,Issue,Status,Recommended_Action
0,AI-001,2026-10-10T08:00:07,Incorrect answer,HIGH,Answer mismatch,Open,Human review required; verify answer and source.
1,AI-002,2026-10-10T08:00:07,Missing citation,MEDIUM,Missing citation,Open,Human review required; verify answer and source.
2,AI-003,2026-10-10T08:00:07,Wrong citation,MEDIUM,Incorrect citation,Open,Human review required; verify answer and source.


In [8]:
from google.colab import files

incident_log.to_csv("ai_agent_incidents.csv", index=False)

files.download("ai_agent_incidents.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
def summarize_incident(row):
    return (
        f"Issue: {row['Issue']}. "
        f"Severity: {row['Severity']}. "
        f"Recommended action: {row['Recommended_Action']}. "
        f"Current status: {row['Status']}."
    )

incident_log["Incident_Summary"] = incident_log.apply(
    summarize_incident, axis=1
)

display(
    incident_log[
        ["Incident_ID", "Severity", "Issue", "Incident_Summary"]
    ]
)

,Incident_ID,Severity,Issue,Incident_Summary
0,AI-001,HIGH,Answer mismatch,Issue: Answer mismatch. Severity: HIGH. Recomm...
1,AI-002,MEDIUM,Missing citation,Issue: Missing citation. Severity: MEDIUM. Rec...
2,AI-003,MEDIUM,Incorrect citation,Issue: Incorrect citation. Severity: MEDIUM. R...


In [10]:
from google.colab import files

incident_log.to_csv("ai_agent_incidents.csv", index=False)
files.download("ai_agent_incidents.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>